In [5]:
import os

In [6]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(override=True)

True

In [7]:
load_dotenv(override=True)
g_api_key = os.getenv('GROQ_API_KEY')

# Check the key

if not g_api_key:
    print("No API key was found!")
elif not g_api_key.startswith("gsk_Voj"):
    print("An API key was found, but it doesn't start gsk_Voj;")
elif g_api_key.strip() != g_api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end")
else:
    print("API key found and looks good so far!")

An API key was found, but it doesn't start gsk_Voj;


In [8]:
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print("No API key was found!")
elif not api_key.startswith("sk-proj"):
    print("An API key was found, but it doesn't start sk-proj-;")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [9]:
from pathlib import Path

In [10]:
from llama_index.core import SimpleDirectoryReader

In [11]:
documents = SimpleDirectoryReader(input_files=['./data/pdf/policydocument.pdf']).load_data()

In [12]:
# Check the datatype and length of the loaded documents
type(documents)

list

In [13]:
# total number of pages read from the PDF
len(documents)

99

In [14]:
documents[0]

Document(id_='147ae7ac-947c-4141-b4da-36f54384fe38', embedding=None, metadata={'page_label': '1', 'file_name': 'policydocument.pdf', 'file_path': 'data/pdf/policydocument.pdf', 'file_type': 'application/pdf', 'file_size': 7245552, 'creation_date': '2026-06-14', 'last_modified_date': '2026-06-14'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text=' \nPage 1 of 99 Policy No. 4C5384296068206854Form No. 754\nDate : 29 Mar, 2026\n : 748194476\n : 4C538429606\n : SBI Life - Smart Shield PlusProduct Name\nPolicy No.\nCustomer No.\nTo,\nSpeedPost/JI803963186IN//\nAURANGABAD /4CYK739297\nUIN  : 111N150V01\n8206854 / 7188 / No / SpeedPost / Indian / 2

In [15]:
from llama_index.core.node_parser import SentenceSplitter

In [16]:
my_splitter=SentenceSplitter(
    chunk_size=1000,
    chunk_overlap=50
)

In [17]:
nodes=my_splitter.get_nodes_from_documents(documents)

In [18]:
len(nodes)

112

In [24]:
from llama_index.embeddings.openai import OpenAIEmbedding

In [25]:
##my_embd_model=OpenAIEmbedding(model="text-embedding-3-small")

In [19]:
from llama_index.core import Settings

In [20]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [21]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

2026-06-16 21:05:19,899 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-16 21:05:19,901 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-16 21:05:19,916 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-06-16 21:05:20,152 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-06-16 21:05:20,307 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-06-16 21:05:20,312 - INFO - Loading SentenceTransformer model from BAAI/b

In [37]:
import chromadb

In [38]:
from llama_index.vector_stores.chroma import ChromaVectorStore

In [39]:
from llama_index.core import StorageContext

In [31]:
from llama_index.core import VectorStoreIndex

In [42]:
# initialize client, setting path to save data
import chromadb

db = chromadb.PersistentClient(path="./chroma_db")

try:
    db.delete_collection("my_collection")
except:
    pass

collection = db.create_collection("my_collection")

In [43]:
# create collection
##my_chroma_collection = db.get_or_create_collection("quickstart")

In [44]:
my_vector_store = ChromaVectorStore(chroma_collection=collection)

In [45]:
my_storage_context = StorageContext.from_defaults(vector_store=my_vector_store)

In [46]:
# create your index
my_index = VectorStoreIndex.from_documents(nodes, embed_model=Settings.embed_model, storage_context=my_storage_context)

In [47]:
my_retriever=my_index.as_retriever()

nodes=my_retriever.retrieve("what is Better Half Benefit")

In [48]:
len(nodes)

2

In [49]:
for node in nodes:
  print("Node Id:",node.id_)
  print("Metadata:",node.metadata)
  print("Score:",node.get_score())
  print("------------------------------------")

Node Id: 2f5076d9-c952-4084-b9a3-e94d0eb6552d
Metadata: {'page_label': '27', 'file_name': 'policydocument.pdf', 'file_path': 'data/pdf/policydocument.pdf', 'file_type': 'application/pdf', 'file_size': 7245552, 'creation_date': '2026-06-14', 'last_modified_date': '2026-06-14'}
Score: 0.5377130469498774
------------------------------------
Node Id: 621606d7-1761-42e0-af49-7022533fa65c
Metadata: {'page_label': '31', 'file_name': 'policydocument.pdf', 'file_path': 'data/pdf/policydocument.pdf', 'file_type': 'application/pdf', 'file_size': 7245552, 'creation_date': '2026-06-14', 'last_modified_date': '2026-06-14'}
Score: 0.5294300959602165
------------------------------------


In [55]:
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4o-mini")

In [56]:
from llama_index.core import get_response_synthesizer

In [57]:
my_rs=get_response_synthesizer(llm=llm, response_mode="compact")

In [58]:
my_qe=my_index.as_query_engine(llm=llm,response_synthesizer=my_rs)

In [59]:
response=my_qe.query("what is Better Half Benefit")

2026-06-18 22:17:42,906 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [61]:
from IPython.display import display, Markdown
display(Markdown(response.response))

Better Half Benefit is an optional feature available under the SBI Life – Smart Shield Plus policy. It provides additional coverage for the spouse of the Life Assured in the event of the Life Assured's death during the policy term, provided the policy is in force and the spouse is alive. The benefits include:

1. Payment of the Death Benefit for the Life Assured.
2. Commencement of a life cover for the spouse, which lasts for the remaining policy term or until the spouse reaches the age of 60, whichever comes first.
3. No future premiums are required after the Life Assured's death.
4. Upon the spouse's death, a lump sum payment of the Better Half Benefit Sum Assured is made.

This benefit is subject to certain exclusions, such as simultaneous death of the Life Assured and spouse, the spouse reaching age 60, or death due to suicide within 12 months of the Life Assured's death. The spouse named at the policy's inception cannot be changed during the policy term.

In [62]:
response=my_qe.query("explain me the benefits of this term insurance policy in 10 bullet points")

2026-06-18 22:22:26,996 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [63]:
from IPython.display import display, Markdown
display(Markdown(response.response))

1. **Death Benefit**: The policy provides a Sum Assured on Death, which is the highest of several calculated amounts, ensuring financial support to beneficiaries.

2. **Better Half Benefit**: An optional feature that offers life cover for the spouse of the Life Assured upon their death, continuing for the remaining policy term or until the spouse turns 60.

3. **No Future Premiums After Death**: If the Life Assured passes away, no further premiums are required under the policy.

4. **Lumpsum Payment for Spouse's Death**: In the event of the spouse's death, a specified Better Half Benefit Sum Assured is paid in a lump sum.

5. **Flexible Death Benefit Payment Modes**: Nominees can choose to receive the Death Benefit as a lump sum, in installments, or a combination of both.

6. **Minimum Installment Amounts**: The policy specifies minimum amounts for installment payments, ensuring a structured payout.

7. **Level Cover for Better Half Benefit**: The Better Half Benefit provides a consistent level of coverage, regardless of the plan option chosen by the Life Assured.

8. **Non-Linked and Non-Participating**: The policy is a pure risk product, meaning it does not participate in profits and is not linked to any investment component.

9. **Revival Option**: The policy can be revived within five years of non-payment of premiums, restoring all benefits upon fulfilling certain conditions.

10. **Defined Policy Terms**: Clear definitions and terms regarding premium payment, policy year, and surrender ensure transparency and understanding of the policy's workings.

In [72]:
response=my_qe.query("Proposer and Life Assured Details is he Smoker / Non Smoker status of the Life Assured ?")

2026-06-18 22:28:40,027 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [73]:
from IPython.display import display, Markdown
display(Markdown(response.response))

The provided information does not include details regarding the smoker or non-smoker status of the Life Assured.

Use Reursive Retriever to more better contextual and llm response

In [74]:
# The nest_asyncio module enables the nesting of asynchronous functions within an already running async loop.
# This is necessary because Jupyter notebooks inherently operate in an asynchronous loop.
# By applying nest_asyncio, we can run additional async functions within this existing loop without conflicts.
import nest_asyncio

nest_asyncio.apply()

In [93]:
import camelot
import fitz 
import pandas as pd
from typing import List

In [77]:
from llama_index.core import Document

In [78]:
from llama_index.core import VectorStoreIndex

In [79]:
from llama_index.core.query_engine import RetrieverQueryEngine

In [81]:
from llama_index.core.retrievers import RecursiveRetriever
from llama_index.core.schema import IndexNode
from llama_index.llms.openai import OpenAI
from typing import List

In [83]:
from llama_index.readers.file import PyMuPDFReader

In [84]:
loader = PyMuPDFReader()
docs = loader.load("./data/pdf/policydocument.pdf")

In [86]:
docs[0]

Document(id_='9cddcad9-36ff-41e0-89ff-2777752db780', embedding=None, metadata={'total_pages': 99, 'file_path': './data/pdf/policydocument.pdf', 'source': '1'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text=' \nPage 1 of 99\nPolicy No. 4C538429606\n8206854\nForm No. 754\nDate : 29 Mar, 2026\n : 748194476\n : 4C538429606\n : SBI Life - Smart Shield Plus\nProduct Name\nPolicy No.\nCustomer No.\nTo,\nSpeedPost/JI803963186IN//\nAURANGABAD /4CYK739297\nUIN\n : 111N150V01\n8206854 / 7188 / No / SpeedPost / Indian / 255 / EV3\nMr. Rajiv  Ranjan\nCHANDRA NANDA KUTEER 813/D2\nFLAT NO D3-06 3RD FLOOR BTM LAYOUT\n2ND STAGE BANGALORE\nBANGALORE - 560076\nKARNATAKA, India\nBooked under\nBNPL Account Number –\nNMR/DA-NM/NAF – SBI LIFE/1135/24-25\nWELCOME LETTER\nContact Details : 091 9164595991\nDear Mr. Rajiv Ranjan,\nWe welcome You to the SBI 

In [94]:
import fitz
import camelot
import pandas as pd
from typing import List

def get_all_tables(pdf_path: str) -> List[pd.DataFrame]:

    total_pages = len(fitz.open(pdf_path))

    tables = camelot.read_pdf(
        pdf_path,
        pages=f"1-{total_pages}",
        flavor="lattice"
    )

    table_dfs = []

    for table in tables:
        df = table.df

        if len(df) > 1:
            df = (
                df.rename(columns=df.iloc[0])
                .drop(df.index[0])
                .reset_index(drop=True)
            )

        table_dfs.append(df)

    return table_dfs

In [95]:
tables = get_all_tables("./data/pdf/policydocument.pdf")

In [96]:
len(tables)

125

,"Part A SBI Life – Smart Shield Plus (UIN:111N150V01) \n An Individual, Non-Linked, Non-Participating, Life Insurance, Pure Risk Product \n \n D\neclaration by the Policyholder \n I\n have received the above and I have read and confirm having noted the details. \n P\nlace: \n D\nate: \n (\nSignature of the Policyholder) \n N\note: \ni. \nProduct \nrelated documents \nincluding \nthe Customer \nInformation sheet are available on Our website \nwww.sbilife.co.in. \nii. \nIn case of any conflict, the terms and conditions mentioned in the Policy Document shall prevail.",,,,,
0,,,,"AURANGABAD, SBI LIFE INSURANCE CO LTD CST\nNO....",,
1,,14.,Grievances \n/Complaints, \nContact details of Grievance Redressal Off...,Part C (V),
2,,,,,,


In [99]:
first_20_tables = tables[:20]

In [100]:
len(first_20_tables)

20

In [106]:
# This will create 6 pandas query engine. these query engines help us to query these tables in natural language with the help of LLM.
from llama_index.experimental.query_engine import PandasQueryEngine

In [107]:
df_query_engines = [
    PandasQueryEngine(table, llm=llm)
    for table in tables
]

In [ ]:
response = df_query_engines[0].query(
    "What's Lapse, Paid-Up and Revival of the Policy?"
)
print(str(response))